Recall, diaries, and the first visit
====================================

**Author:** Ethan Ligon



## What this is



Session 2 showed that GLSS7's first food visit records about 20% more
expenditure than the five that follow, and argued that this is telescoping
rather than better enumeration.  That argument was made in four lines of
pandas and it deserves more scrutiny than four lines can carry.

This notebook is that scrutiny.  The question underneath it is practical:
you are about to build a consumption aggregate out of these visits, and you
have to decide what to do with the first one.  Dropping it throws away a
sixth of your data.  Keeping it builds a known bias into every poverty
number you report afterwards.  Neither is obviously right.

-   **Prerequisites:** `lsms_library` on the release kernel, GhanaLSS microdata.



## Setup



In [1]:
%xmode Plain

import lsms_library as ll
import numpy as np, pandas as pd

ghana = ll.Country('GhanaLSS')
glss7 = ghana.food_acquired().xs('2016-17', level='t').xs('purchased', level='s')

byvisit = pd.DataFrame({
    'exp':   glss7.groupby(['i', 'visit'])['Expenditure'].sum(),
    'items': glss7.groupby(['i', 'visit']).size(),
})
byvisit.groupby('visit').mean().round(2)

## 1.  Is it the same households?



The lecture compared the mean at visit 2 against the mean over visits 3 to

1.  Those are means over different sets of households: 13,733 at visit 2 and

fewer at each later visit.  If the households that drop out are the ones who
spend least, the decline is composition, not telescoping.



In [1]:
counts = byvisit.groupby('visit').size()
print(counts.to_string())

**Exercise 1.1.** Restrict to the balanced panel of households present at all
six visits and redo the comparison.  How much of the 20.6% survives?  This
is the first thing a referee will ask and it takes three lines.

**Exercise 1.2.** Now go the other way.  Among households *missing* from later
visits, was their visit-2 expenditure unusually high or low?  Say what that
implies about the direction of the bias in exercise 1.1.



## 1.  Telescoping, or a longer first window?



The lecture's claim was that visit 2 carries larger amounts against roughly
the same number of items, which reads as telescoping.  There is a duller
explanation that would produce exactly the same pattern: the first visit may
simply cover more days than five.



In [1]:
v1 = sorted(byvisit.index.get_level_values('visit').unique())[0]  # first food visit
first, rest = byvisit.xs(v1, level='visit'), byvisit.drop(v1, level='visit')
for col in ('exp', 'items'):
    print(f"{col:6s} first food visit vs the other five: "
          f"{100 * (first[col].mean() / rest[col].mean() - 1):+.1f}%")

The form, [Section 9B](https://hhsurveys.ligonresearch.org/hub/user-redirect/files/reading/GLSS7-2016-17-Section9B-food-expenditure.pdf)
(PDF, opens in a new tab), asks "how much was spent and what quantity was
acquired … since my last visit?", and never names a number of days.
  So the window is whatever the enumerator's schedule made
it, and the gap before the first expenditure visit is the one least likely
to have been five days.

**Exercise 2.1.** If the first visit covered $d$ days rather than 5, both
expenditure and item count should scale with $d$, and expenditure per item
should not move at all.  The data say expenditure is up 20.6% and items up
4.5%, so expenditure per item *is* up.  Write down what $d$ would have to
be to explain the item count, and show that the same $d$ cannot explain
the expenditure.  This is the argument the lecture asserted; make it.

**Exercise 2.2.** A third story: enumerators are keenest on their first call
and the diary is freshest.  That would raise items and expenditure together.
Distinguish it from telescoping using the *composition* of what is reported:
telescoping should pull in lumpy, infrequent purchases, while enthusiasm
should add small frequent ones.  Look at the distribution of item-level
values at visit 2 against later visits, not just the mean.



## 1.  Does it matter for anything you would report?



The reason to care is that the aggregate is built from these visits.

**Exercise 3.1.** Build household food expenditure for 2016-17 three ways:
all six visits; visits 3 to 7 only, scaled by 6/5; and all six with visit 2
replaced by the household's own mean over visits 3 to 7.  Report the
weighted poverty headcount at a fixed line under each.  How far apart are
they?  Compare that spread against the 19 percentage points Beegle et al.
found across module designs.

**Exercise 3.2.** Redo 3.1 for the Gini and for Atkinson at
$\varepsilon = 1$.  Session 2 claimed rankings are more robust than levels.
Is that true here?

**Exercise 3.3.** Having done both, say which construction you would publish,
and write the two sentences of justification you would put in a footnote.
This is the actual deliverable; everything above is working.



## 1.  Does the pattern hold in the earlier rounds?



GLSS6 (`2012-13`) and GLSS5 (`2005-06`) also used repeat visits.



In [1]:
acq = ghana.food_acquired()
(acq.xs('purchased', level='s')
    .groupby(['t', 'visit'])['Expenditure']
    .mean()
    .unstack('visit')
    .round(1))

**Exercise 4.1.** Is the first-visit premium a GLSS7 artefact or a feature of
the instrument?  If it appears in every round, that is evidence about how
people answer diaries.  If it appears only in GLSS7, look at what changed:
the item list went from 114 to 179 between GLSS6 and GLSS7, which session 2
flagged for other reasons.

**Exercise 4.2.** The 1987-88 and 1988-89 rounds record their recall period as
"since my last visit", which is not a fixed window at all.  Show what that
does to any attempt to put those rounds on the same axis as GLSS7, and say
whether you would include them in a long-run series of Ghanaian food
consumption.



## Exercises



1.  Everything above is about *purchased* food, because that is what carries
    an expenditure.  Own production is recorded against the same visits.
    Does it show the same first-visit premium?  If it does not, that is
    informative about the mechanism, since you cannot telescope a harvest
    you ate.
2.  Section 3 asked you to pick a construction.  Now suppose the statistical
    office has already published the all-six-visits number and you are
    writing a comment.  What is the smallest defensible claim you can make
    about the direction of the error in the published poverty rate, and what
    would you need that you do not have to make a claim about its size?

